## 1. Loading the data

Two files:

| File | Rows | Contains |
|---|---|---|
| `data/donnees_demandes.csv` | 10,000 | historical applications **with** `decision_octroi` |
| `data/candidats_evaluation.csv` | 4,000 | new applications, **without** labels - these are the ones you'll be scored on |


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    selection_rate,
    true_positive_rate,
)

demandes = pd.read_csv('data/donnees_demandes.csv')
candidats = pd.read_csv('data/candidats_evaluation.csv')

print(f'{len(demandes)} dossiers historiques, {len(candidats)} a evaluer')
demandes.head()


10000 dossiers historiques, 4000 a evaluer


,id_candidat,cote_r_equivalent,programme_etudes,region_administrative,code_postal_3,revenu_familial_estime,heures_travail_semaine,distance_domicile_campus_km,premiere_generation_universitaire,decision_octroi
0,C008033,25.96,Genie,Cote-Nord,G4R,17017,9,168.0,0,0
1,C004299,26.82,Arts et lettres,Cote-Nord,G5B,66820,22,99.3,0,0
2,C011788,27.37,Genie,Bas-Saint-Laurent,G5N,49223,19,364.1,0,0
3,C001144,21.23,Sciences,Gaspesie-Iles-de-la-Madeleine,G5C,90750,17,174.8,1,0
4,C003322,28.27,Sante,Bas-Saint-Laurent,G5N,36892,9,151.2,0,0


In [2]:
demandes.info()
print()
print(demandes.describe(include='all').T[['count', 'unique', 'top', 'mean']])


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id_candidat                        10000 non-null  str    
 1   cote_r_equivalent                  10000 non-null  float64
 2   programme_etudes                   10000 non-null  str    
 3   region_administrative              10000 non-null  str    
 4   code_postal_3                      10000 non-null  str    
 5   revenu_familial_estime             10000 non-null  int64  
 6   heures_travail_semaine             10000 non-null  int64  
 7   distance_domicile_campus_km        10000 non-null  float64
 8   premiere_generation_universitaire  10000 non-null  int64  
 9   decision_octroi                    10000 non-null  int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 781.4 KB

                                     count unique       top        mean
id

## 2. A first look at the regions

The committee awarded a scholarship to 40% of applicants. Is that rate the same everywhere?


In [3]:
# Les cinq regions administratives presentes dans le jeu de donnees.
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']

def groupe_region(df):
    """Regroupe les cinq regions en deux blocs : grands centres / regions eloignees."""
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Eloignee', 'Centre')

par_region = demandes.groupby('region_administrative').agg(
    candidats=('decision_octroi', 'size'),
    taux_octroi=('decision_octroi', 'mean'),
    cote_r_moyenne=('cote_r_equivalent', 'mean'),
).sort_values('taux_octroi')
print(par_region.round(3))

demandes['groupe'] = groupe_region(demandes)
print()
print(demandes.groupby('groupe')['decision_octroi'].mean().round(3))


                               candidats  taux_octroi  cote_r_moyenne
region_administrative                                                
Cote-Nord                           1178        0.263          27.325
Bas-Saint-Laurent                   1293        0.269          27.296
Gaspesie-Iles-de-la-Madeleine       1529        0.284          27.354
Montreal                            4001        0.483          27.988
Capitale-Nationale                  1999        0.484          27.984

groupe
Centre      0.484
Eloignee    0.273
Name: decision_octroi, dtype: float64


Note the gap in R score between regions, then the gap in award rate.
Is the second fully explained by the first? Keep the question open.


## 3. The production model

A random forest trained on every available column, including
`region_administrative`. This is the model the institution uses today.


In [4]:
CATEGORIELLES = ['programme_etudes', 'region_administrative', 'code_postal_3']

def encoder(df_entrainement, df_cible):
    """Encode les colonnes categorielles et aligne les colonnes des deux tableaux.

    L'alignement (reindex) est important : si une categorie est absente d'un des
    deux jeux, get_dummies produit un nombre de colonnes different et le modele
    echoue - ou pire, apprend sur les mauvaises colonnes.
    """
    X = pd.get_dummies(
        df_entrainement.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    )
    Xc = pd.get_dummies(
        df_cible.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    ).reindex(columns=X.columns, fill_value=0)
    return X, Xc

# Decoupage interne pour mesurer honnetement la performance.
train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)
X_train, X_test = encoder(train, test)
y_train, y_test = train['decision_octroi'], test['decision_octroi']

modele = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print(f'Exactitude globale : {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, digits=3))


Exactitude globale : 88.13%

              precision    recall  f1-score   support

           0      0.890     0.916     0.903      1802
           1      0.867     0.830     0.848      1198

    accuracy                          0.881      3000
   macro avg      0.879     0.873     0.875      3000
weighted avg      0.881     0.881     0.881      3000



Roughly 88% accuracy. As far as the quarterly dashboard is concerned, the model is doing great.


## 4. Fairness audit

Two measures, and they don't say the same thing:

- **Demographic parity** - do both groups receive scholarships at the same rate?
- **Equality of opportunity** - among the applicants who deserve the scholarship, do both
  groups have the same chance of getting it?

The second requires knowing who deserves the scholarship. Look closely at what we're using to compute it here.


In [5]:
groupe_test = groupe_region(test)

cadre = MetricFrame(
    metrics={
        'exactitude': accuracy_score,
        'taux_selection': selection_rate,
        'taux_vrais_positifs': true_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=groupe_test,
)
print(cadre.by_group.round(4))

ecart_parite = demographic_parity_difference(y_test, y_pred, sensitive_features=groupe_test)
print(f'\nEcart de parite demographique : {ecart_parite:.4f}')


                     exactitude  taux_selection  taux_vrais_positifs
sensitive_feature_0                                                 
Centre                   0.8777          0.4586               0.8475
Eloignee                 0.8865          0.2710               0.7847

Ecart de parite demographique : 0.1876


The `taux_vrais_positifs` (true positive rate) above is computed against `decision_octroi`,
that is, against the committee's past decisions.

In other words: we've just verified that the model faithfully reproduces the committee.
That is not the same thing as verifying that it is fair.


## 5. The trap

The reflex is to drop `region_administrative` and declare the problem solved.
Let's measure.


In [6]:
essais = {'modele complet': []}

def entrainer_sans(colonnes_exclues, etiquette):
    X, Xt = encoder(train, test)
    garder = [c for c in X.columns if not c.startswith(tuple(colonnes_exclues))] if colonnes_exclues else list(X.columns)
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
    m.fit(X[garder], y_train)
    p = m.predict(Xt[garder])
    ecart = demographic_parity_difference(y_test, p, sensitive_features=groupe_test)
    print(f'{etiquette:<42} exactitude {accuracy_score(y_test, p):.4f}   ecart parite {ecart:.4f}')
    return ecart

entrainer_sans((), 'toutes les colonnes')
entrainer_sans(('region_administrative_',), 'sans region_administrative')
entrainer_sans(('region_administrative_', 'code_postal_3_'), 'sans region ni code postal')


toutes les colonnes                        exactitude 0.8813   ecart parite 0.1876
sans region_administrative                 exactitude 0.8837   ecart parite 0.1805
sans region ni code postal                 exactitude 0.8797   ecart parite 0.1737


np.float64(0.1736705949985628)

The gap barely moves.

Other columns carry the regional information: home-to-campus distance, hours
worked, family income, postal code. Removing the label does not remove the
signal. This is the **proxy effect**.

Measure it yourself - a correlation matrix, or a model that tries to predict the
region from the other columns, will tell you which ones are the most talkative.


## 6. Your mission

Produce, for the 4,000 applicants in `data/candidats_evaluation.csv`, an award
decision that is fairer than the current model's.

Three constraints:

1. **Fixed budget.** The institution funds a fixed number of scholarships. Your overall
   award rate must stay between **36% and 44%**. A submission outside that range receives 0
   on the technical section. Awarding the scholarship to everyone is not a solution.
2. **You are not scored on `decision_octroi`.** The jury compares your decisions to a
   reference standard built independently of the historical committee, which you do not have.
   Perfectly reproducing `decision_octroi` perfectly reproduces the bias.
3. **Format.** Two columns, 4,000 rows, values 0 or 1.

The next cell writes a valid submission from the biased model. It serves as a
template. Replace `predictions` with your own.


In [7]:
X_hist, X_cand = encoder(demandes, candidats)
modele_complet = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele_complet.fit(X_hist, demandes['decision_octroi'])
predictions = modele_complet.predict(X_cand)

soumission = pd.DataFrame({
    'id_candidat': candidats['id_candidat'],
    'decision_octroi': predictions.astype(int),
})
soumission.to_csv('predictions.csv', index=False)

taux = soumission['decision_octroi'].mean()
print(f'{len(soumission)} lignes ecrites dans predictions.csv')
print(f'Taux d\'octroi : {taux:.1%}', '- OK' if 0.36 <= taux <= 0.44 else '- HORS BUDGET')
soumission.head()


4000 lignes ecrites dans predictions.csv
Taux d'octroi : 39.1% - OK


,id_candidat,decision_octroi
0,C011476,0
1,C000471,0
2,C006911,0
3,C007718,0
4,C003531,0


---

### Directions to explore next

- Which columns best predict the region? Those are your proxies.
- Between demographic parity and equality of opportunity, which would you defend before an
  ethics committee, and why? The two cannot both be satisfied at the same time
  if the two groups do not have exactly the same profile.
- `fairlearn.postprocessing.ThresholdOptimizer` and `fairlearn.reductions.ExponentiatedGradient`
  are the two fastest tools to set up. The first adjusts thresholds after the
  fact, the second retrains under constraint.
- Vary the fairness constraint and plot the (fairness, accuracy) scatter: that's your
  Pareto frontier, and the jury will ask you to defend it.
- If `decision_octroi` is a biased label, is the best model the one that
  predicts it best?
